# Konya | Model comparison, explanations and valuation grid
CME434 · Karabük University

Review the stored nested spatial cross-validation, validate the grid, and reproduce the fit when needed.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ART = ROOT / 'artifacts'


In [ ]:
report = json.loads((ART / 'model_evaluation.json').read_text(encoding='utf8'))
scores = pd.DataFrame({name: card['metrics'] for name, card in report['models'].items()}).T
display(scores.sort_values('rmse'))
assert scores.rmse.idxmin() == report['best_model']
display(pd.DataFrame(report['folds']).drop(columns='test_blocks'))


In [ ]:
import rasterio
import numpy as np

with rasterio.open(ART / 'valuation_grid_1km.tif') as source:
    raster = source.read(masked=True)
    assert source.crs.to_epsg() == 32636 and source.res == (1000, 1000)
    assert raster[0].count() == 40771
    assert np.isclose(np.ma.median(raster[1]), 100)

display(pd.read_csv(ART / 'shap_summary.csv').head(10))
display(Image(filename=str(ART / 'figures' / '03_konya_valuation_grid.png'),width=870))


In [ ]:
# Set to True to regenerate fitted estimators, SHAP values, and maps in this checkout.
REBUILD = False
if REBUILD:
    from train import run
    from make_figures import main
    run()
    main()
